## Demo RAG offline từ file export

Notebook này mô phỏng kịch bản triển khai biên (edge deployment): chỉ sử dụng file export
(`GET /api/documents/{id}/export`) cùng Ollama chạy cục bộ, không gọi tới `RagHub.API`.

Lưu ý: các vector trong file export gắn với một mô hình embedding cụ thể. Nếu đổi mô hình,
vector cũ không còn dùng được và phải embed lại từ trường `content`.

In [10]:
import json
import numpy as np
import requests

BUNDLE_PATH = "samples/sample_export.json"
OLLAMA_BASE = "http://localhost:11434"
GENERATION_MODEL = "gemma4:e4b"
NUM_CTX = 8192  
TOP_K = 4

In [11]:
with open(BUNDLE_PATH, encoding="utf-8") as f:
    bundle = json.load(f)

embedding_model = bundle["embeddingModel"]
embedding_dim = bundle["embeddingDim"]
chunks = bundle["chunks"]

# Đảm bảo mọi chunk đã có embedding trước khi tính cosine similarity,
# tránh lỗi lệch shape khó truy vết ở bước search phía dưới.
missing = [c for c in chunks if not c.get("embedding")]
assert not missing, f"{len(missing)} chunk thiếu embedding — tài liệu có thể chưa ở trạng thái Indexed khi export"

matrix = np.array([c["embedding"] for c in chunks], dtype=np.float32)
assert matrix.shape[1] == embedding_dim, f"file khai báo dim={embedding_dim} nhưng vector thực tế có {matrix.shape[1]} chiều"

print(bundle["name"], "-", len(chunks), "chunks, model:", embedding_model)

cam_nang_sau_dai_hoc_2025_0.pdf - 94 chunks, model: bge-m3


### Truy hồi (retrieval)

Dùng cosine similarity dạng brute-force trên toàn bộ ma trận chunk, không xây chỉ mục ANN.
Với quy mô vài chục trang tài liệu, cách tính trực tiếp này vẫn nhanh hơn round-trip gọi
API embedding, và đơn giản hơn cho mục đích demo.

In [12]:
def embed_query(text: str) -> np.ndarray:
    resp = requests.post(f"{OLLAMA_BASE}/api/embed", json={"model": embedding_model, "input": [text]}, timeout=60)
    resp.raise_for_status()
    vec = resp.json()["embeddings"][0]
    if len(vec) != embedding_dim:
        # Cùng tên model nhưng khác phiên bản/mức lượng tử hoá (quantization) sẽ cho
        # số chiều embedding khác nhau — kiểm tra sớm để tránh lỗi khó chẩn đoán về sau.
        raise ValueError(f"model cục bộ trả về {len(vec)} chiều, trong khi file export yêu cầu {embedding_dim} chiều — không cùng model")
    return np.array(vec, dtype=np.float32)


def search(query: str, top_k: int = TOP_K):
    q = embed_query(query)
    sims = matrix @ q / (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q) + 1e-8)
    top_idx = np.argsort(-sims)[:top_k]
    return [(chunks[i], float(sims[i])) for i in top_idx]

In [15]:
query = "UIT có mấy cơ sơ?"

results = search(query)
for chunk, score in results:
    print(f"[{score:.3f}] {chunk['headingPath'] or '(no heading)'} - trang {chunk['pageNo']}")
    print("  ", chunk["content"][:200].replace("\n", " "), "...")

[0.520] (no heading) - trang 26
   3725-2002- số nội bộ: 110. Địa chỉ: P. A106, Khu phố 6, Phường Linh Trung, TP. Thủ Đức, TP. HCM. 23  TT Nội dung Chuyên viên phụ trách 1 Quản lý đào tạo trình độ ThS. Nguyễn Thị Diễm Thúy thạc sĩ (đề  ...
[0.510] (no heading) - trang 25
   en.uit.edu.vn/news/Lists/tinhoatdong/DispFor m.aspx?PageIndex=0&ID=250&InitialTabId=Ribbon.Re ad 22  ● Làm thẻ thư viện, thẻ HVCH, Cấp tài khoản và email cá nhân cho từng HVCH để sử dụng trong quá trì ...
[0.457] (no heading) - trang 81
   sinh CTĐT thạc sĩ. Học viên thỏa nhiều mức nhận học bổng thì chỉ được nhận học bổng mức cao nhất. 5  Mức học STT Điều kiện tối thiểu bổng (1) Bài báo tạp chí/hội nghị thuộc danh sách UIT -XS VÀ: • B ...
[0.427] (no heading) - trang 103
   Quy chế này do Hiệu trưởng Trường ĐH CNTT xem xét, quyết định./. HIỆU TRƯỞNG Nguyễn Hoàng Tú Anh 31  DANH MỤC TỪ VIẾT TẮT Bộ GD&ĐT: Bộ Giáo dục và Đào tạo CSĐT: cơ sở đào tạo CTĐT: chương trình đào tạ ...


### Sinh câu trả lời (generation)

Ghép context từ các chunk truy hồi được rồi gọi `/api/generate`. Logic tương đương
`ChatService` ở backend, viết lại dạng rút gọn để tiện thử nghiệm trong notebook.

In [16]:
def assemble_prompt(query: str, results) -> str:
    context = "\n\n".join(
        f"[chunk {c['chunkIndex']}] ({c['headingPath'] or 'n/a'}, trang {c['pageNo']})\n{c['content']}"
        for c, _ in results
    )
    return (
        f"Chỉ trả lời dựa trên context được cung cấp, không suy diễn thêm thông tin ngoài context. "
        f"Trích rõ số chunk đã sử dụng.\n\n"
        f"Context:\n{context}\n\nCâu hỏi: {query}\nTrả lời:"
    )


def generate(prompt: str):
    resp = requests.post(
        f"{OLLAMA_BASE}/api/generate",
        json={"model": GENERATION_MODEL, "prompt": prompt, "stream": True, "options": {"num_ctx": NUM_CTX}},
        stream=True, timeout=120,
    )
    resp.raise_for_status()
    for line in resp.iter_lines():
        if not line:
            continue
        chunk = json.loads(line)
        if chunk.get("response"):
            print(chunk["response"], end="", flush=True)
        if chunk.get("done"):
            break


generate(assemble_prompt(query, results))

Dựa trên ngữ cảnh được cung cấp, UIT có ít nhất hai cơ sở:
1. Văn phòng tại TP. HCM (VP-NBK): 07-09 Nguyễn Bỉnh Khiêm, Phường Bến Nghé, Quận 1, TP. HCM.
2. Cơ sở Thủ Đức (CS-TĐ): P. A106, Khu phố 6, Phường Linh Trung, TP. Thủ Đức, TP. HCM.

(Sử dụng chunk [21])

---
**Việc cần làm tiếp theo:** thử thêm một số câu hỏi nằm ngoài phạm vi corpus để kiểm tra
xem mô hình có từ chối trả lời ("không tìm thấy thông tin") hay vẫn sinh ra câu trả lời
không dựa trên context (hallucination).